# Clase 10 — Criptografía Moderna y Tendencias

> **Curso:** Criptografía Aplicada  
> **Objetivo general:** Integrar fundamentos modernos de criptografía aplicada (homomorfismo, firmas avanzadas, post-cuántica y JWT) con una visión práctica para ingeniería de sistemas reales.

---

## Tabla de contenidos

1. [Introducción y objetivos](#1)
2. [Panorama actual y motivación](#2)
3. [Importaciones globales](#3)
4. [Criptografía homomórfica (introducción)](#4)
5. [Práctica Python: homomorfismo didáctico](#5)
6. [Firmas digitales avanzadas: Schnorr](#6)
7. [Práctica Python: firma Schnorr completa](#7)
8. [Firmas BLS: intuición y agregación](#8)
9. [Práctica Python: agregación tipo BLS (modelo didáctico)](#9)
10. [Criptografía post-cuántica (NIST)](#10)
11. [Práctica Python: esquema tipo retícula (LWE simplificado)](#11)
12. [Kyber y Dilithium en arquitectura real](#12)
13. [JWT: fundamentos, riesgos y buenas prácticas](#13)
14. [Práctica Python: emisión y verificación de JWT HS256](#14)
15. [Aplicaciones reales y patrones de diseño](#15)
16. [Migración a escenarios híbridos clásico+PQC](#16)
17. [Checklist de hardening criptográfico](#17)
18. [Laboratorio integrado](#18)
19. [Ejercicios propuestos](#19)
20. [Referencias](#20)


<a id='1'></a>
## 1) Introducción y objetivos

La criptografía aplicada en 2026 exige **diseño por capas**:
- Cifrado y autenticación clásicos bien implementados.
- Firmas con propiedades avanzadas (agregación, eficiencia en cadena, identidad).
- Preparación para amenaza cuántica (criptoagilidad).
- Integración segura en protocolos y aplicaciones (API, identidad, IoT, blockchain, banca).

### Objetivos específicos de esta clase
1. Comprender la idea de computación sobre datos cifrados (homomorfismo).
2. Implementar una firma Schnorr educativa y validar su seguridad operacional.
3. Entender el valor de BLS para agregación de firmas.
4. Diferenciar KEM/firmas post-cuánticas y su adopción (Kyber/Dilithium).
5. Construir y auditar JWT de forma segura en Python.


<a id='2'></a>
## 2) Panorama actual y motivación

### ¿Por qué estas tendencias importan?
- **Privacidad computable:** análisis en la nube sin revelar datos en claro.
- **Escalabilidad de verificación:** firmas agregadas para miles de validadores.
- **Riesgo "harvest now, decrypt later":** datos capturados hoy podrían romperse mañana.
- **Identidad y autorización ubicuas:** JWT en microservicios, BFFs, zero trust APIs.

### Principio de ingeniería
No existe una única primitiva "mágica". En práctica se combinan:
- Criptografía simétrica (rápida)
- Criptografía asimétrica (intercambio y firma)
- Derivación de claves
- Controles operativos (rotación, observabilidad, políticas)


<a id='3'></a>
## Importaciones globales


In [ ]:
import base64
import hashlib
import hmac
import json
import math
import os
import random
import secrets
import statistics
import time
from dataclasses import dataclass
from typing import Dict, Any, Tuple, List



<a id='4'></a>
## 4) Criptografía homomórfica (introducción)

La criptografía homomórfica permite computar funciones sobre datos cifrados sin descifrar previamente.

### Tipos (visión breve)
- **PHE (Partially Homomorphic Encryption):** soporta una operación (suma o producto).
- **SHE (Somewhat HE):** soporta varias operaciones con límite de profundidad.
- **FHE (Fully HE):** soporta circuitos arbitrarios (más costosa).

### Casos de uso
- Analítica privada en salud/finanzas.
- Machine learning sobre datos cifrados.
- Procesamiento en terceros no confiables.

> En esta clase usaremos demos **didácticas** para entender propiedades, no implementaciones productivas de FHE.


<a id='5'></a>
## 5) Práctica Python: homomorfismo didáctico

### 5.1 Esquema aditivo juguete (NO seguro)
Definimos cifrado modular: `Enc(m)= (m + r) mod N`, con `r` aleatorio guardado como parte del ciphertext.  
Se preserva suma modular al combinar componentes, útil para observar homomorfismo.


In [ ]:
# Esquema didáctico aditivo (NO usar en producción)
N = 9973

def toy_add_encrypt(m: int) -> Tuple[int, int]:
    r = secrets.randbelow(N)
    c = (m + r) % N
    return c, r

def toy_add_decrypt(ct: Tuple[int, int]) -> int:
    c, r = ct
    return (c - r) % N

def toy_add_homomorphic_add(ct1: Tuple[int, int], ct2: Tuple[int, int]) -> Tuple[int, int]:
    c1, r1 = ct1
    c2, r2 = ct2
    return ((c1 + c2) % N, (r1 + r2) % N)

m1, m2 = 123, 456
ct1 = toy_add_encrypt(m1)
ct2 = toy_add_encrypt(m2)
ct_sum = toy_add_homomorphic_add(ct1, ct2)

print('m1 dec =', toy_add_decrypt(ct1))
print('m2 dec =', toy_add_decrypt(ct2))
print('dec(ct1 (+) ct2) =', toy_add_decrypt(ct_sum))
print('(m1 + m2) mod N =', (m1 + m2) % N)


In [ ]:
# Mini experimento: suma agregada de salarios anonimizados (didáctico)
salarios = [1200, 1500, 1700, 2100, 1900]
ciphertexts = [toy_add_encrypt(x) for x in salarios]
agg = (0, 0)
for ct in ciphertexts:
    agg = toy_add_homomorphic_add(agg, ct)

suma_descifrada = toy_add_decrypt(agg)
print('Suma real:', sum(salarios) % N)
print('Suma desde cifrados:', suma_descifrada)
print('Coinciden:', suma_descifrada == (sum(salarios) % N))


### 5.2 Limitaciones prácticas reales
- Coste computacional muy alto en HE real (latencia/CPU/memoria).
- Restricciones de precisión (aproximaciones, ruido criptográfico).
- Requiere diseño de pipeline completo (serialización, parámetros, evaluación segura).

Herramientas del ecosistema (para estudiar): Microsoft SEAL, OpenFHE, PALISADE (histórico), TenSEAL.


<a id='6'></a>
## 6) Firmas digitales avanzadas: Schnorr

Schnorr ofrece firmas compactas y un diseño elegante sobre grupos de logaritmo discreto.

### Idea base
- Clave privada: `x`
- Clave pública: `X = g^x mod p`
- Firma (esquema simplificado): `(R, s)`
- Verificación comprueba una relación algebraica con desafío hash.

### Ventajas
- Simplicidad matemática.
- Base para construcciones multi-firma.
- Seguridad bien estudiada en modelo aleatorio con supuestos estándar.


<a id='7'></a>
## 7) Práctica Python: firma Schnorr completa

> Implementación educativa sobre grupo modular pequeño para visualización. En producción se usan curvas elípticas y estándares robustos.


In [ ]:
# Schnorr didáctico en Z_p*
P = 2089
G = 2
Q = P - 1  # simplificación didáctica e INSEGURA: en Schnorr real Q es el orden primo grande de un subgrupo


def h_challenge(R: int, X: int, msg: bytes) -> int:
    data = f'{R}|{X}|'.encode() + msg
    return int.from_bytes(hashlib.sha256(data).digest(), 'big') % Q


def schnorr_keygen() -> Tuple[int, int]:
    x = secrets.randbelow(Q - 2) + 1
    X = pow(G, x, P)
    return x, X


def schnorr_sign(msg: bytes, x: int, X: int) -> Tuple[int, int]:
    k = secrets.randbelow(Q - 2) + 1
    R = pow(G, k, P)
    e = h_challenge(R, X, msg)
    s = (k + e * x) % Q
    return R, s


def schnorr_verify(msg: bytes, sig: Tuple[int, int], X: int) -> bool:
    R, s = sig
    e = h_challenge(R, X, msg)
    lhs = pow(G, s, P)
    rhs = (R * pow(X, e, P)) % P
    return lhs == rhs

sk, pk = schnorr_keygen()
message = b'Clase10: firma Schnorr demo'
sig = schnorr_sign(message, sk, pk)

print('Public key:', pk)
print('Signature:', sig)
print('Verify original:', schnorr_verify(message, sig, pk))
print('Verify tampered:', schnorr_verify(message + b'X', sig, pk))


In [ ]:
# Prueba por lotes: tasa de verificación
ok = 0
NTEST = 100
for i in range(NTEST):
    msg = f'msg-{i}'.encode()
    sig_i = schnorr_sign(msg, sk, pk)
    ok += int(schnorr_verify(msg, sig_i, pk))

print(f'Firmas válidas: {ok}/{NTEST}')


<a id='8'></a>
## 8) Firmas BLS: intuición y agregación

BLS (Boneh–Lynn–Shacham) habilita **firmas cortas** y agregación eficiente.

### Valor operacional
- Múltiples firmantes → una firma agregada.
- Verificación compacta en redes distribuidas.
- Muy utilizada en investigación/protocolos blockchain.

### Nota
La implementación real de BLS requiere emparejamientos bilineales en curvas específicas (BLS12-381). Aquí veremos una simulación educativa de agregación semántica.


<a id='9'></a>
## 9) Práctica Python: agregación tipo BLS (modelo didáctico)

Esta simulación conserva la intuición: cada firmante produce una firma y el agregador combina resultados.


In [ ]:
# Modelo didáctico de agregación de firmas (NO BLS real)
PRIME = 2**255 - 19
GEN = 5


def H(msg: bytes) -> int:
    return int.from_bytes(hashlib.sha256(msg).digest(), 'big') % PRIME


def toy_bls_keygen() -> Tuple[int, int]:
    sk = secrets.randbelow(PRIME - 2) + 1
    pk = (sk * GEN) % PRIME
    return sk, pk


def toy_bls_sign(msg: bytes, sk: int) -> int:
    return (H(msg) * sk) % PRIME


def toy_bls_aggregate(signatures: List[int]) -> int:
    return sum(signatures) % PRIME


def toy_bls_verify_aggregate_same_msg(msg: bytes, agg_sig: int, pks: List[int]) -> bool:
    lhs = agg_sig % PRIME
    rhs = (H(msg) * sum(pks)) % PRIME
    return lhs == rhs

msg = b'Bloque #1024'
keys = [toy_bls_keygen() for _ in range(8)]
sigs = [toy_bls_sign(msg, sk) for sk, _ in keys]
agg_sig = toy_bls_aggregate(sigs)

print('Agregada valida (mismo mensaje):', toy_bls_verify_aggregate_same_msg(msg, agg_sig, [pk for _, pk in keys]))


In [ ]:
# Sensibilidad: un mensaje distinto invalida la agregación esperada
sigs_bad = sigs.copy()
sigs_bad[0] = toy_bls_sign(b'Bloque #OTRO', keys[0][0])
agg_bad = toy_bls_aggregate(sigs_bad)
print('Agregada con firmante alterado:', toy_bls_verify_aggregate_same_msg(msg, agg_bad, [pk for _, pk in keys]))


<a id='10'></a>
## 10) Criptografía post-cuántica (NIST)

### Contexto
La estandarización post-cuántica del NIST prioriza primitivas resistentes a ataques con computación cuántica escalable.

### Familias clave
- **KEM (intercambio de claves):** CRYSTALS-Kyber (ML-KEM en estandarización reciente).
- **Firmas digitales:** CRYSTALS-Dilithium (ML-DSA), Falcon, SPHINCS+.

### Impacto de ingeniería
- Claves y firmas suelen ser más grandes que en ECC/RSA.
- Mayor presión sobre ancho de banda, almacenamiento y HSM/KMS.
- Necesidad de **criptoagilidad** para rotar algoritmos sin rediseñar toda la plataforma.


<a id='11'></a>
## 11) Práctica Python: esquema tipo retícula (LWE simplificado)

> Demo pedagógica para intuición de "ruido" en problemas de retículas. No representa seguridad real de Kyber.


In [ ]:
# LWE simplificado 1D (didáctico)
MOD = 4093


def lwe_keygen() -> Tuple[int, int]:
    s = secrets.randbelow(MOD)
    a = secrets.randbelow(MOD)
    return s, a


def sample_small_error(bound: int = 2) -> int:
    return secrets.randbelow(2 * bound + 1) - bound


def lwe_encrypt_bit(bit: int, a_pub: int, b_pub: int) -> Tuple[int, int]:
    r = secrets.randbelow(MOD)
    e1 = sample_small_error()
    e2 = sample_small_error()
    u = (a_pub * r + e1) % MOD
    v = (b_pub * r + e2 + (MOD // 2 if bit else 0)) % MOD
    return u, v


def lwe_decrypt_bit(cipher: Tuple[int, int], s: int) -> int:
    u, v = cipher
    x = (v - s * u) % MOD
    # umbral simple alrededor de MOD/2
    return 1 if abs(x - MOD // 2) < MOD // 4 else 0

# clave pública estilo (a, b=a*s+e)
s, a = lwe_keygen()
e = sample_small_error()
b = (a * s + e) % MOD

acc = 0
NTEST = 200
for _ in range(NTEST):
    bit = secrets.randbelow(2)
    c = lwe_encrypt_bit(bit, a, b)
    dec = lwe_decrypt_bit(c, s)
    acc += int(bit == dec)

print('Exactitud de descifrado (demo):', round(acc / NTEST, 3))


In [ ]:
# Perfil de tamaños aproximados (conceptual)
approx_sizes = {
    'RSA-3072 pubkey': '≈ 384 bytes',
    'P-256 pubkey': '≈ 64 bytes',
    'Kyber pubkey (orden de magnitud)': '≈ kilobytes',
    'Dilithium firma (orden de magnitud)': '≈ kilobytes'
}
for k, v in approx_sizes.items():
    print(f'{k:40s} -> {v}')


<a id='12'></a>
## 12) Kyber y Dilithium en arquitectura real

### Diseño recomendado de transición
1. **Modo híbrido de intercambio:** ECDHE + Kyber KEM (concatenación y HKDF).
2. **Doble firma en periodos críticos:** clásica + post-cuántica según política.
3. **Versionado criptográfico:** incluir `alg`, `kdf`, `kem`, `sig` explícitos en metadatos.
4. **Telemetría criptográfica:** medir tamaños, latencia, fallos por algoritmo.

### Riesgos frecuentes
- Downgrade de algoritmo por negociación débil.
- Validación incompleta de claves/certificados.
- Rotación asimétrica (clientes actualizados, servidores no).


In [ ]:
# Mini patrón de secreto híbrido: clásico + "pqc" simulado + HKDF

def hkdf_extract(salt: bytes, ikm: bytes) -> bytes:
    return hmac.new(salt, ikm, hashlib.sha256).digest()


def hkdf_expand(prk: bytes, info: bytes, length: int = 32) -> bytes:
    t = b''
    okm = b''
    counter = 1
    while len(okm) < length:
        t = hmac.new(prk, t + info + bytes([counter]), hashlib.sha256).digest()
        okm += t
        counter += 1
    return okm[:length]

shared_classic = hashlib.sha256(b'classic-ecdhe-secret').digest()
shared_pqc = hashlib.sha256(b'kyber-shared-secret-simulado').digest()

salt = os.urandom(16)
prk = hkdf_extract(salt, shared_classic + shared_pqc)
app_key = hkdf_expand(prk, b'handshake hybrid key', 32)

print('Hybrid key (hex):', app_key.hex())


<a id='13'></a>
## 13) JWT: fundamentos, riesgos y buenas prácticas

JWT (JSON Web Token) es un formato compacto para transportar claims firmados.

### Estructura
`base64url(header).base64url(payload).base64url(signature)`

### Riesgos clásicos
- Aceptar `alg=none`.
- Confundir algoritmos simétricos/asimétricos.
- No validar `exp`, `nbf`, `iss`, `aud`.
- No rotar claves (`kid`, JWKS) ni manejar revocación.

### Recomendaciones
- Allowlist estricta de algoritmos.
- TTL corto y refresh tokens controlados.
- Claims mínimos (evitar datos sensibles).
- Firma y validación desacopladas con observabilidad.


<a id='14'></a>
## 14) Práctica Python: emisión y verificación de JWT HS256


In [ ]:
# JWT HS256 sin librerías externas (educativo)

def b64url_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b'=').decode()


def b64url_decode(data: str) -> bytes:
    pad = '=' * (-len(data) % 4)
    return base64.urlsafe_b64decode(data + pad)


def jwt_sign_hs256(payload: Dict[str, Any], secret: bytes, header: Dict[str, Any] | None = None) -> str:
    if header is None:
        header = {'alg': 'HS256', 'typ': 'JWT'}
    h = b64url_encode(json.dumps(header, separators=(',', ':')).encode())
    p = b64url_encode(json.dumps(payload, separators=(',', ':')).encode())
    signing_input = f'{h}.{p}'.encode()
    sig = hmac.new(secret, signing_input, hashlib.sha256).digest()
    return f'{h}.{p}.{b64url_encode(sig)}'


def jwt_verify_hs256(token: str, secret: bytes, issuer: str, audience: str, leeway: int = 0) -> Dict[str, Any]:
    parts = token.split('.')
    if len(parts) != 3:
        raise ValueError('Formato JWT inválido')

    h_b64, p_b64, s_b64 = parts
    header = json.loads(b64url_decode(h_b64))

    if header.get('alg') != 'HS256':
        raise ValueError('Algoritmo no permitido')

    signing_input = f'{h_b64}.{p_b64}'.encode()
    expected = hmac.new(secret, signing_input, hashlib.sha256).digest()
    got = b64url_decode(s_b64)

    if not hmac.compare_digest(expected, got):
        raise ValueError('Firma inválida')

    payload = json.loads(b64url_decode(p_b64))
    now = int(time.time())

    if 'exp' in payload and now > int(payload['exp']) + leeway:
        raise ValueError('Token expirado')
    if 'nbf' in payload and now + leeway < int(payload['nbf']):
        raise ValueError('Token aún no válido (nbf)')
    if payload.get('iss') != issuer:
        raise ValueError('Issuer inválido')
    if payload.get('aud') != audience:
        raise ValueError('Audience inválido')

    return payload

secret = os.urandom(32)
now = int(time.time())
payload = {
    'sub': 'user-123',
    'role': 'admin',
    'iss': 'https://auth.ejemplo.local',
    'aud': 'api://pagos',
    'iat': now,
    'nbf': now,
    'exp': now + 120,
    'jti': secrets.token_hex(8)
}

tok = jwt_sign_hs256(payload, secret)
print('JWT:', tok[:90] + '...')
claims = jwt_verify_hs256(tok, secret, issuer='https://auth.ejemplo.local', audience='api://pagos')
print('Claims verificados:', claims)


In [ ]:
# Ataque de manipulación de payload (debe fallar)
parts = tok.split('.')
header = json.loads(b64url_decode(parts[0]))
payload2 = json.loads(b64url_decode(parts[1]))
payload2['role'] = 'superadmin'

tampered_payload = b64url_encode(json.dumps(payload2, separators=(',', ':')).encode())
tampered = f"{parts[0]}.{tampered_payload}.{parts[2]}"

try:
    jwt_verify_hs256(tampered, secret, issuer='https://auth.ejemplo.local', audience='api://pagos')
    print('ERROR: no debió validar')
except Exception as e:
    print('Verificación correcta (rechazado):', e)


<a id='15'></a>
## 15) Aplicaciones reales y patrones de diseño

### Escenario A: API Gateway + JWT + mTLS interno
- El proveedor de identidad firma token.
- Gateway valida firma, emite contexto mínimo.
- Servicios internos confían en identidad reenviada por canal autenticado.

### Escenario B: Blockchain/consenso
- Firmas avanzadas reducen ancho de banda de validación.
- Agregación ayuda en bloques con muchos validadores.

### Escenario C: Datos sensibles en nube
- Preprocesamiento homomórfico selectivo para estadísticas sobre datos cifrados.
- Estrategia híbrida con enclaves/TEE según carga y riesgo.


<a id='16'></a>
## 16) Migración a escenarios híbridos clásico+PQC

### Hoja de ruta sugerida
1. Inventario criptográfico (algoritmos, librerías, certificados, KMS).
2. Clasificación de datos por horizonte de confidencialidad.
3. Piloto híbrido en un flujo crítico de bajo riesgo.
4. Métricas de impacto (latencia, tamaño handshake, CPU, errores).
5. Plan de rollback y feature flags por algoritmo.

### Señales de madurez
- Criptoagilidad integrada en CI/CD.
- Pruebas de downgrade y compatibilidad cruzada.
- Rotación automatizada de material criptográfico.


<a id='17'></a>
## 17) Checklist de hardening criptográfico

- [ ] Algoritmos y tamaños mínimos alineados con estándar vigente.
- [ ] Nonces/IVs únicos por clave y modo de operación.
- [ ] Validación estricta de tokens/firmas/certificados.
- [ ] Rotación y versionado de claves (`kid`, expiración, revocación).
- [ ] Registro de eventos criptográficos (fallos de verificación, uso de algoritmos legacy).
- [ ] Pruebas de carga con parámetros post-cuánticos.
- [ ] Política de dependencia segura (SBOM, CVEs, actualización continua).


<a id='18'></a>
## 18) Laboratorio integrado

### Objetivo
Diseñar una mini arquitectura segura para una fintech con:
1. Autenticación JWT para APIs.
2. Firma de transacciones de alto riesgo.
3. Plan de transición a KEM post-cuántico híbrido.

### Actividades
1. Emitir token de acceso con claims mínimos y TTL corto.
2. Firmar una "orden" con Schnorr didáctico y validar.
3. Derivar clave de sesión híbrida con HKDF usando dos secretos.
4. Medir impacto de tamaño de artefactos (comparativa conceptual).
5. Elaborar plan de rotación trimestral de claves.


In [ ]:
# Ejecución rápida del laboratorio integrado
orden = b'transferir:1000:USD:dest=ACME'

# 1) Token
lab_secret = os.urandom(32)
lab_now = int(time.time())
lab_token = jwt_sign_hs256({
    'sub': 'cliente-77',
    'scope': 'transfer:write',
    'iss': 'https://auth.fintech.local',
    'aud': 'api://core-banking',
    'iat': lab_now,
    'nbf': lab_now,
    'exp': lab_now + 60
}, lab_secret)

# 2) Firma Schnorr educativa
lab_sk, lab_pk = schnorr_keygen()
lab_sig = schnorr_sign(orden, lab_sk, lab_pk)
lab_ok_sig = schnorr_verify(orden, lab_sig, lab_pk)

# 3) Secreto híbrido
h1 = hashlib.sha256(b'classic-secret-lab').digest()
h2 = hashlib.sha256(b'pqc-secret-lab').digest()
lab_key = hkdf_expand(hkdf_extract(os.urandom(16), h1 + h2), b'fintech-session', 32)

# 4) Verificar token
lab_claims = jwt_verify_hs256(
    lab_token,
    lab_secret,
    issuer='https://auth.fintech.local',
    audience='api://core-banking'
)

print('JWT válido para:', lab_claims['sub'])
print('Firma de orden válida:', lab_ok_sig)
print('Session key (len):', len(lab_key), 'bytes')


<a id='19'></a>
## 19) Ejercicios propuestos

1. Modificar la demo de homomorfismo para calcular promedio cifrado de una lista.
2. Implementar derivación determinista de nonce para Schnorr y comparar riesgos.
3. Simular agregación de firmas con 1, 10, 100 y 1000 firmantes (tiempo/tamaño).
4. Extender JWT validator para soportar `kid` y un JWKS local simulado.
5. Diseñar una política de criptoagilidad para una empresa con 50 microservicios.
6. Construir pruebas de seguridad negativas: `exp` vencido, `aud` inválida, token truncado.
7. Proponer estrategia de migración híbrida para datos con retención de 10 años.


<a id='20'></a>
## 20) Referencias

### Estándares y documentos técnicos
- NIST Post-Quantum Cryptography Project: https://csrc.nist.gov/projects/post-quantum-cryptography
- FIPS 203 (ML-KEM / Kyber) y FIPS 204 (ML-DSA / Dilithium) — publicaciones NIST vigentes.
- RFC 7519 — JSON Web Token (JWT): https://datatracker.ietf.org/doc/html/rfc7519
- RFC 8725 — JWT Best Current Practices: https://datatracker.ietf.org/doc/html/rfc8725
- RFC 5869 — HKDF: https://datatracker.ietf.org/doc/html/rfc5869

### Papers y fundamentos
- C. Schnorr (1991), *Efficient Identification and Signatures for Smart Cards*.
- Boneh, Lynn, Shacham (2001), *Short Signatures from the Weil Pairing*.
- Regev (2005), *On Lattices, Learning with Errors, Random Linear Codes, and Cryptography*.
- Craig Gentry (2009), *A Fully Homomorphic Encryption Scheme*.

### Recursos aplicados
- OWASP Cheat Sheet — Cryptographic Storage.
- OWASP Cheat Sheet — JWT for Java/General Security Guidance.
- Microsoft SEAL / OpenFHE documentation for homomorphic encryption experimentation.

---
**Nota pedagógica:** las implementaciones de esta clase son demostrativas y priorizan comprensión conceptual. Para entornos productivos, usar bibliotecas auditadas, parámetros estandarizados y validación criptográfica formal.
